In [ ]:
# 1. Setup de caminhos locais
import os
import subprocess
from datetime import datetime
from pathlib import Path

from src.io import paths
from src.config import loader as cfg

DATA_PATHS = paths.get_data_paths()
PROJECT_PATHS = paths.get_project_paths()
BASE_PATH = str(DATA_PATHS["base"])
RAW_PATH = str(DATA_PATHS["data_raw"])
PROC_PATH = str(DATA_PATHS["data_processed"])
INTERIM_PATH = str(DATA_PATHS["data_interim"])
CONFIG = cfg.load_config()

NB_NAME = "01_preprocessing"
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

print("BASE_PATH:", BASE_PATH)
print("PROC_PATH:", PROC_PATH)


In [ ]:
# 3. Carregar IBOV
import pandas as pd, os
ibov_file = os.path.join(RAW_PATH, "ibovespa.csv")
assert os.path.exists(ibov_file), f"Arquivo nao encontrado: {ibov_file}"
df_ibov = pd.read_csv(ibov_file)

date_aliases = ["data", "day", "date"]
date_col = next((col for col in date_aliases if col in df_ibov.columns), None)
if date_col is None:
    raise KeyError("ibovespa.csv precisa de uma coluna de data (data/day/date)")
df_ibov["data"] = pd.to_datetime(df_ibov[date_col])
if date_col != "data":
    df_ibov = df_ibov.drop(columns=[date_col])

if "close" not in df_ibov.columns:
    if "adj_close" in df_ibov.columns:
        df_ibov["close"] = df_ibov["adj_close"]
    else:
        raise KeyError("ibovespa.csv precisa da coluna close ou adj_close")

print("IBOV:", df_ibov.shape)
display(df_ibov.head())


In [ ]:
# 4. Carregar Notícias (múltiplas fontes possíveis)
import pandas as pd

# Tentar carregar de múltiplas fontes (ordem de preferência)
news_sources = [
    os.path.join(RAW_PATH, "noticias_real.csv"),           # Gerado pelo NB 05
    os.path.join(PROC_PATH, "news_clean.parquet"),         # Gerado pelo NB 13
    os.path.join(PROC_PATH, "noticias_real_clean.csv"),    # Alternativa
    os.path.join(RAW_PATH, "noticias_exemplo.csv"),        # Demo (última opção)
]

df_news = None
loaded_from = None

for news_file in news_sources:
    if os.path.exists(news_file):
        try:
            if news_file.endswith('.parquet'):
                df_news = pd.read_parquet(news_file)
            else:
                df_news = pd.read_csv(news_file)
            loaded_from = news_file
            break
        except Exception as e:
            print(f"⚠️ Erro ao ler {news_file}: {e}")
            continue

if df_news is None:
    print("\n❌ ERRO: Nenhum arquivo de notícias encontrado!")
    print("\nPara executar este notebook, primeiro execute:")
    print("  - Notebook 05 (data_collection_real) para coletar notícias reais via NewsAPI")
    print("  - OU Notebook 12+13 (multisource + ETL) para pipeline completo")
    print("\nArquivos procurados (ordem de preferência):")
    for f in news_sources:
        print(f"  - {f}")
    raise FileNotFoundError("Nenhuma fonte de notícias disponível")

# Normalizar colunas
if "data" not in df_news.columns:
    # Tentar aliases comuns
    date_aliases = ["day", "date", "Data", "publishedAt"]
    date_col = next((col for col in date_aliases if col in df_news.columns), None)
    if date_col:
        df_news["data"] = pd.to_datetime(df_news[date_col])
    else:
        raise KeyError(f"Arquivo de notícias precisa de coluna de data. Colunas disponíveis: {list(df_news.columns)}")
else:
    df_news["data"] = pd.to_datetime(df_news["data"])

# Normalizar coluna de texto
if "texto" not in df_news.columns:
    text_aliases = ["text", "texto", "description", "title"]
    text_col = next((col for col in text_aliases if col in df_news.columns), None)
    if text_col:
        df_news["texto"] = df_news[text_col].fillna("")
    else:
        raise KeyError(f"Arquivo de notícias precisa de coluna de texto. Colunas disponíveis: {list(df_news.columns)}")

print(f"✅ Notícias carregadas de: {os.path.basename(loaded_from)}")
print(f"   Shape: {df_news.shape}")
print(f"   Período: {df_news['data'].min()} → {df_news['data'].max()}")
display(df_news.head())


In [ ]:
# 5. Pré-processamento de texto
import re, nltk
nltk.download("stopwords")
from nltk.corpus import stopwords

stop_pt = set(stopwords.words("portuguese"))
PATTERN = r"[^A-Za-z\u00C0-\u00FF\s]"

def preprocess_text(t):
    t = re.sub(PATTERN, " ", str(t)).lower()
    toks = [w for w in t.split() if w and w not in stop_pt]
    return " ".join(toks)

# Usar a coluna 'texto'
df_news["clean_text"] = df_news["texto"].apply(preprocess_text)

print("✅ Pré-processamento concluído!")
display(df_news[["texto", "clean_text"]].head())


In [ ]:
from pathlib import Path
print("Arquivos em PROC_PATH:")
for path in Path(PROC_PATH).iterdir():
    size_kb = path.stat().st_size / 1024
    print(f" - {path.name} ({size_kb:0.1f} KB)")

In [ ]:
# 6. Validar período de cobertura
PERIODO = cfg.get_periodo_estudo()
START_EXPECTED = pd.to_datetime(PERIODO["start"])
END_EXPECTED = pd.to_datetime(PERIODO["end"])

if "data" in df.columns:
    min_date = pd.to_datetime(df["data"].min())
    max_date = pd.to_datetime(df["data"].max())
    print(f"\n=== Validação de Período ===")
    print(f"Esperado: {START_EXPECTED.date()} → {END_EXPECTED.date()}")
    print(f"Obtido:   {min_date.date()} → {max_date.date()}")
    
    if min_date > START_EXPECTED + pd.Timedelta(days=30):
        print(f"⚠️ AVISO: Início dos dados ({min_date.date()}) está >30 dias após o esperado.")
    if max_date < END_EXPECTED - pd.Timedelta(days=30):
        print(f"⚠️ AVISO: Fim dos dados ({max_date.date()}) está >30 dias antes do esperado.")
    if min_date <= START_EXPECTED and max_date >= END_EXPECTED:
        print("✅ Período validado com sucesso!")
else:
    print("⚠️ Coluna 'data' não encontrada para validação de período.")